# 08. End-to-end тестирование и сценарии

Этот ноутбук помогает описать тестирование веб-сервиса в магистерской работе. Здесь проверяется backend service и бизнес-формулы для трёх пользовательских сценариев. Для ручной проверки frontend дополнительно приведены команды запуска.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

from backend.app.schemas import MovieInput
from backend.app.services.predictor import predict_movie_success

## Тестовые сценарии

Сценарии являются условными. Они не утверждают реальные прогнозы для настоящих фильмов, а проверяют корректность прохождения данных через backend и расчёта бизнес-показателей.

In [ ]:
scenarios = [
    {
        "scenario": "Высокобюджетный коммерческий фильм",
        "payload": {
            "title": "Условный блокбастер",
            "rating": "PG-13",
            "genre": "Action",
            "year": 2024,
            "score": 6.5,
            "votes": 150000,
            "director": "Режиссёр",
            "writer": "Сценарист",
            "star": "Ведущий актёр",
            "country": "США",
            "budget": 120000000,
            "company": "Large Studio",
            "runtime": 130,
            "released": "2024",
        },
    },
    {
        "scenario": "Среднебюджетный фильм",
        "payload": {
            "title": "Условная драма",
            "rating": "16+",
            "genre": "Drama",
            "year": 2024,
            "score": 6.5,
            "votes": 35000,
            "director": "Режиссёр",
            "writer": "Сценарист",
            "star": "Актёр",
            "country": "Россия",
            "budget": 12000000,
            "company": "Film Company",
            "runtime": 105,
            "released": "2024",
        },
    },
    {
        "scenario": "Рискованный фильм с низким бюджетом",
        "payload": {
            "title": "Условный независимый фильм",
            "rating": "18+",
            "genre": "Drama",
            "year": 2024,
            "score": 6.5,
            "votes": 2000,
            "director": "Новый режиссёр",
            "writer": "Новый сценарист",
            "star": "Новый актёр",
            "country": "Россия",
            "budget": 500000,
            "company": "Independent Studio",
            "runtime": 95,
            "released": "2024",
        },
    },
]

In [ ]:
results = []
for item in scenarios:
    movie_input = MovieInput(**item["payload"])
    prediction = predict_movie_success(movie_input)
    data = prediction.model_dump() if hasattr(prediction, "model_dump") else prediction.dict()
    results.append({
        "scenario": item["scenario"],
        "predicted_revenue": data["predicted_revenue"],
        "predicted_profit": data["predicted_profit"],
        "roi": data["roi"],
        "payback_ratio": data["payback_ratio"],
        "success_category": data["success_category"],
        "risk_level": data["risk_level"],
    })

results_df = pd.DataFrame(results)
results_df

## Проверка формул по сценариям

Для каждого сценария проверяется:

- `predicted_profit = predicted_revenue - budget`;
- `roi = predicted_profit / budget`;
- `payback_ratio = predicted_revenue / budget`;
- `payback_ratio = roi + 1`.

In [ ]:
formula_checks = []
for item, result in zip(scenarios, results):
    budget = item["payload"]["budget"]
    formula_checks.append({
        "scenario": item["scenario"],
        "profit_formula_ok": abs(result["predicted_profit"] - (result["predicted_revenue"] - budget)) < 1e-6,
        "roi_formula_ok": abs(result["roi"] - (result["predicted_profit"] / budget)) < 1e-6,
        "payback_formula_ok": abs(result["payback_ratio"] - (result["predicted_revenue"] / budget)) < 1e-6,
        "payback_equals_roi_plus_one": abs(result["payback_ratio"] - (result["roi"] + 1)) < 1e-6,
    })

pd.DataFrame(formula_checks)

## Команды ручной проверки интерфейса

Backend:

```bash
uvicorn backend.app.main:app --reload
```

Frontend:

```bash
cd frontend
npm run dev
```

Открыть страницу:

```text
http://127.0.0.1:5173/
```

После заполнения формы пользователь сначала проверяет данные, затем нажимает `Рассчитать прогноз`, после чего форма исчезает и открывается отчёт.

## Вывод

End-to-end проверка подтверждает, что сервис возвращает все необходимые бизнес-показатели и что формулы расчёта согласованы.